# 02 - PyTorch Regularization and Generalization A/B Tests

This notebook mirrors the TensorFlow notebook but uses PyTorch idioms.

## Topics covered
- L2 regularization via weight decay
- manual L1 penalty
- dropout
- early stopping
- Monte Carlo dropout
- initialization comparisons
- batch normalization
- custom dropout module
- TensorBoard logging

The goal is not just to train a model, but to compare how each technique changes optimization and generalization.


In [ ]:
# Colab setup
%pip -q install -U torch torchvision tensorboard pandas matplotlib scikit-learn


In [ ]:
import os
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## Dataset

We use FashionMNIST again so the PyTorch and TensorFlow A/B tests are directly comparable.


In [ ]:
transform = transforms.ToTensor()
train_full = datasets.FashionMNIST(root="data", train=True, download=True, transform=transform)
test_set = datasets.FashionMNIST(root="data", train=False, download=True, transform=transform)

# shrink for faster Colab runs
train_subset = torch.utils.data.Subset(train_full, list(range(15000)))
test_subset = torch.utils.data.Subset(test_set, list(range(3000)))

train_len = 12000
val_len = len(train_subset) - train_len
train_set, val_set = random_split(train_subset, [train_len, val_len], generator=torch.Generator().manual_seed(SEED))

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
val_loader = DataLoader(val_set, batch_size=256, shuffle=False)
test_loader = DataLoader(test_subset, batch_size=256, shuffle=False)

len(train_set), len(val_set), len(test_subset)


In [ ]:
class StochasticScaleDropout(nn.Module):
    def __init__(self, rate=0.2):
        super().__init__()
        self.rate = rate

    def forward(self, x):
        if self.training:
            noise = torch.empty_like(x).uniform_(1.0 - self.rate, 1.0 + self.rate)
            return x * noise
        return x

class MLP(nn.Module):
    def __init__(self, hidden_sizes=(256, 128), dropout=0.0, batchnorm=False, custom_dropout=False):
        super().__init__()
        layers_list = [nn.Flatten()]
        in_dim = 28 * 28

        for hidden in hidden_sizes:
            layers_list.append(nn.Linear(in_dim, hidden))
            if batchnorm:
                layers_list.append(nn.BatchNorm1d(hidden))
            layers_list.append(nn.ReLU())
            if dropout > 0:
                if custom_dropout:
                    layers_list.append(StochasticScaleDropout(dropout))
                else:
                    layers_list.append(nn.Dropout(dropout))
            in_dim = hidden

        layers_list.append(nn.Linear(in_dim, 10))
        self.net = nn.Sequential(*layers_list)

    def forward(self, x):
        return self.net(x)


In [ ]:
def initialize_weights(model, init_name="xavier"):
    for module in model.modules():
        if isinstance(module, nn.Linear):
            if init_name == "xavier":
                nn.init.xavier_uniform_(module.weight)
            elif init_name == "he":
                nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
            elif init_name == "lecun_like":
                nn.init.normal_(module.weight, mean=0.0, std=np.sqrt(1.0 / module.in_features))
            nn.init.zeros_(module.bias)

def l1_penalty(model):
    return sum(param.abs().sum() for name, param in model.named_parameters() if "weight" in name)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total = 0
    correct = 0
    losses = []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        losses.append(loss.item())
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    return np.mean(losses), correct / total

def train_model(name, config, epochs=10):
    model = MLP(
        dropout=config.get("dropout", 0.0),
        batchnorm=config.get("batchnorm", False),
        custom_dropout=config.get("custom_dropout", False),
    ).to(device)
    initialize_weights(model, config.get("initializer", "xavier"))

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.get("lr", 1e-3),
        weight_decay=config.get("weight_decay", 0.0),
    )

    writer = SummaryWriter(log_dir=os.path.join("runs", "part1_torch", name))

    best_state = None
    best_val_acc = -1
    patience = 3
    patience_counter = 0
    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(epochs):
        model.train()
        batch_losses = []

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits, y)

            if config.get("l1_strength", 0.0) > 0:
                loss = loss + config["l1_strength"] * l1_penalty(model)

            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())

        train_loss = float(np.mean(batch_losses))
        val_loss, val_acc = evaluate(model, val_loader)

        writer.add_scalar("loss/train", train_loss, epoch)
        writer.add_scalar("loss/val", val_loss, epoch)
        writer.add_scalar("acc/val", val_acc, epoch)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"[{name}] early stopping at epoch {epoch + 1}")
            break

    model.load_state_dict(best_state)
    test_loss, test_acc = evaluate(model, test_loader)
    writer.close()

    result = {
        "experiment": name,
        "best_val_acc": best_val_acc,
        "test_acc": test_acc,
        "epochs_ran": len(history["train_loss"]),
    }
    return model, history, result


## Core A/B tests


In [ ]:
configs = {
    "baseline": {},
    "weight_decay_l2": {"weight_decay": 1e-4},
    "manual_l1": {"l1_strength": 1e-6},
    "dropout": {"dropout": 0.30},
    "batchnorm": {"batchnorm": True},
    "dropout_batchnorm": {"dropout": 0.30, "batchnorm": True},
    "custom_dropout": {"dropout": 0.20, "custom_dropout": True},
}

torch_models = {}
torch_histories = {}
rows = []

for name, config in configs.items():
    print("Running", name)
    model, history, row = train_model(name, config, epochs=10)
    torch_models[name] = model
    torch_histories[name] = history
    rows.append(row)

torch_results = pd.DataFrame(rows).sort_values("test_acc", ascending=False)
torch_results


In [ ]:
plt.figure(figsize=(10, 5))
for name, history in torch_histories.items():
    plt.plot(history["val_acc"], label=name)
plt.title("PyTorch validation accuracy by experiment")
plt.xlabel("Epoch")
plt.ylabel("val_accuracy")
plt.legend()
plt.show()


## Initialization A/B tests

A rough rule of thumb:

- **Kaiming / He** for ReLU stacks
- **Xavier / Glorot** as a broad default
- **LeCun-like** when you want smaller variance scaling

This is not sacred scripture from the machine gods; it is a practical starting point.


In [ ]:
init_rows = []
for init_name in ["xavier", "he", "lecun_like"]:
    model, history, row = train_model(
        f"init_{init_name}",
        {"initializer": init_name, "dropout": 0.2},
        epochs=8,
    )
    init_rows.append(row)

pd.DataFrame(init_rows).sort_values("test_acc", ascending=False)


## Monte Carlo dropout

To use MC dropout in PyTorch, we keep dropout layers active during inference by flipping them back into train mode while avoiding gradient updates.


In [ ]:
mc_model, mc_history, mc_row = train_model("mc_dropout", {"dropout": 0.3}, epochs=10)
mc_row


In [ ]:
def enable_dropout_only(model):
    model.eval()
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.train()

enable_dropout_only(mc_model)

x_batch, y_batch = next(iter(test_loader))
x_batch = x_batch[:16].to(device)
y_batch = y_batch[:16]

mc_probs = []
with torch.no_grad():
    for _ in range(25):
        probs = torch.softmax(mc_model(x_batch), dim=1)
        mc_probs.append(probs.cpu().numpy())

mc_probs = np.stack(mc_probs)
mean_probs = mc_probs.mean(axis=0)
std_probs = mc_probs.std(axis=0)
preds = mean_probs.argmax(axis=1)
uncertainty = std_probs.max(axis=1)

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax, image, true_y, pred_y, u in zip(axes.ravel(), x_batch.cpu(), y_batch, preds, uncertainty):
    ax.imshow(image.squeeze(), cmap="gray")
    ax.set_title(f"T:{int(true_y)} P:{int(pred_y)}\nunc={u:.3f}")
    ax.axis("off")
plt.tight_layout()
plt.show()


## TensorBoard

PyTorch writes logs through `SummaryWriter`. In Colab, point TensorBoard at the `runs/` directory.


In [ ]:
print("TensorBoard log root:", os.path.abspath("runs/part1_torch"))
# In Colab:
# %load_ext tensorboard
# %tensorboard --logdir runs/part1_torch


## Takeaways

- `weight_decay` gives you L2 regularization with almost no extra code.
- Manual L1 is easy to add directly into the loss.
- Dropout, batch normalization, and early stopping all change generalization in different ways.
- Initialization matters because bad starting scales can poison gradient flow from the first step.
- MC dropout gives uncertainty estimates without training a full ensemble.

This notebook pairs nicely with the TensorFlow notebook for a clean framework comparison in the final video.
